In [14]:
import pandas as pd
import numpy as np
import geopandas as gpd
import geodatasets
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# enable latex plotting 
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

import constants as c 
from logger import setup_logger 
log = setup_logger("generate-flood-risk-coverage-maps")
log.setLevel("INFO")
log.info("Modules loaded.")

2025-10-19 14:53:56 - generate-flood-risk-coverage-maps - INFO - Modules loaded.


In [15]:
ct_nyc = gpd.read_file(f"{c.GEO_PATH}/ct-nyc-2020.geojson", crs=c.WGS).to_crs(c.PROJ)
log.info("Loaded NYC Census Tracts.")

nybb = gpd.read_file(geodatasets.get_path("nybb"), crs=c.WGS).to_crs(c.PROJ)
log.info("Loaded NYC Boroughs.")

2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Loaded NYC Census Tracts.
2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Loaded NYC Boroughs.


In [16]:
analysis_df = pd.read_csv(c.CURRENT_DF)
log.info("Analysis dataframe loaded.")

# ESTIMATE_THRES should be the 25th quantile of p_y among tracts with confirmed_flooding_image 
analysis_df['confirmed_flooding_image'] = analysis_df['at_least_one_positive_image_by_area'] == 1
log.info(f"Found {analysis_df['confirmed_flooding_image'].sum()} tracts with confirmed flooding images.")

# calculate the 25th quantile of p_y among tracts with confirmed flooding images
ESTIMATE_THRES = analysis_df.loc[analysis_df['confirmed_flooding_image'], 'p_y'].quantile(0.25)
log.info(f"25th quantile of p_y among tracts with confirmed flooding images: {ESTIMATE_THRES}")

2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Analysis dataframe loaded.
2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Found 168 tracts with confirmed flooding images.
2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - 25th quantile of p_y among tracts with confirmed flooding images: 0.006830008944732475


In [17]:
analysis_df['any_sensors'] = analysis_df['n_floodnet_sensors'] > 0
log.info(f"Found {analysis_df['any_sensors'].sum()} tracts with at least one FloodNet sensor.")

# get all columns with 311 in name, and sum them up
analysis_df['n_311_requests'] = analysis_df.filter(like='311').sum(axis=1)
log.info(f"Found {analysis_df['n_311_requests'].sum()} 311 requests.")

analysis_df['any_311_report'] = analysis_df['n_311_requests'] > 0
log.info(f"Found {analysis_df['any_311_report'].sum()} tracts with at least one 311 report.")

analysis_df['no_dep_flooding'] = analysis_df['dep_moderate_2_frac'] == 0
log.info(f"Found {analysis_df['no_dep_flooding'].sum()} tracts with no DEP flooding.")

2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Found 192 tracts with at least one FloodNet sensor.
2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Found 2171 311 requests.
2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Found 878 tracts with at least one 311 report.
2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Found 1001 tracts with no DEP flooding.


In [18]:
# merge geometry from ct_nyc into analysis_df
analysis_df['GEOID'] = analysis_df['GEOID'].astype(str)
analysis_df = ct_nyc.merge(analysis_df, on='GEOID')
# make sure analysis_df has the same number of rows as ct_nyc 
if len(analysis_df) != len(ct_nyc):
    log.error(f"Length of analysis_df ({len(analysis_df)}) does not match length of ct_nyc ({len(ct_nyc)}).")
    exit(1)
else: 
    log.info(f"Length of analysis_df ({len(analysis_df)}) matches length of ct_nyc ({len(ct_nyc)}).")

analysis_df = gpd.GeoDataFrame(analysis_df, crs=c.PROJ)

2025-10-19 14:53:57 - generate-flood-risk-coverage-maps - INFO - Length of analysis_df (2325) matches length of ct_nyc (2325).


In [26]:
import shapely

analysis_df = analysis_df[['geometry','p_y']].to_crs(c.WGS)

# trim geometry to 4 decimal places 
analysis_df['geometry'] = analysis_df['geometry'].apply(lambda geom: shapely.wkt.loads(shapely.wkt.dumps(geom, rounding_precision=4)))

# trim p_y to 3 significant digits 
analysis_df['p_y'] = analysis_df['p_y'].apply(lambda x: round(100*x, 3))


In [27]:
# write to geojson 
analysis_df.to_file('bayflood.geojson')

In [28]:
analysis_df 

,geometry,p_y
0,"MULTIPOLYGON (((-74.04390 40.69020, -74.04350 ...",6.146
1,"POLYGON ((-73.98450 40.70950, -73.98660 40.709...",0.196
2,"POLYGON ((-73.99020 40.71440, -73.98930 40.714...",0.963
3,"POLYGON ((-73.98840 40.71650, -73.98750 40.716...",0.042
4,"POLYGON ((-73.98510 40.71910, -73.98420 40.718...",0.030
...,...,...
2320,"POLYGON ((-74.16140 40.52940, -74.16090 40.528...",0.234
2321,"POLYGON ((-74.16720 40.60210, -74.16760 40.599...",0.600
2322,"POLYGON ((-74.16870 40.62120, -74.16880 40.621...",0.094
2323,"POLYGON ((-73.88200 40.83750, -73.88200 40.837...",0.132
